<a href="https://colab.research.google.com/github/Thundercok/VRPTW-Visualization/blob/main/vrptw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

m# **VRPTW - ALNS + Reinforcement Learning**
**Algorithms:** Standard ALNS , DQN - ALNS

**Dataset:** Solomon Benchmark RC1 + RC2

**Goal:** Show RL-Guided operator selection improves ALNS

# ***1. Setup***

In [15]:
#install, imports
%pip install kagglehub numba safetensors -q

from collections import deque
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from numba import njit

import os, sys, glob, time, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from safetensors.torch import save_file, load_file

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {DEVICE} | Pytorch: {torch.__version__}')

Device: cpu | Pytorch: 2.10.0+cpu


In [16]:
#Download dataset
import kagglehub, shutil

path = kagglehub.dataset_download('senju14/vrptw-benchmark-datasets')
BASE_PATH ='/content/solomon'

if os.path.exists(BASE_PATH):
  shutil.rmtree(BASE_PATH)
shutil.copytree(path, BASE_PATH)

DATA_PATH = '/content/solomon/data/Solomon'
print(f'Dataset path: {DATA_PATH}')
print(f'Files: {len(glob.glob(os.path.join(DATA_PATH, "*.txt")))}')

Using Colab cache for faster access to the 'vrptw-benchmark-datasets' dataset.
Dataset path: /content/solomon/data/Solomon
Files: 56


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/vrptw_results'
os.makedirs(SAVE_DIR, exist_ok=True)

# ***2. Config***

In [17]:
@dataclass
class Config:
  #ALNS
  iterations:           int   = 25_000
  destroy_ratio_min:    float = 0.10
  destroy_ratio_max:    float = 0.40
  temp_control:         float = 0.05            #T0 = tmp_control * cost0 / ln2
  temp_min:             float = 0.001
  temp_decay:         float = 0.99975
  sigma1:               int   = 33              # New global best
  sigma2:               int   = 9               # better than current
  sigma3:               int   = 3               # Accepted (worse)
  sigma4:               int   = 1               # Rejected
  weight_decay:         float = 0.10
  segment_size:         int   = 100
  early_stop_patience:  int   = 5_000
  n_runs:               int   = 3

  #RL
  state_dim:            int   = 12
  hidden_dim:           int   = 128
  lr:                   float = 1e-3
  gamma:                float = 0.95
  epsilon_start:        float = 0.40
  epsilon_end:          float = 0.01
  epsilon_decay:        float = 0.9997
  buffer_size:          int   = 8_000
  batch_size:           int   = 64
  target_update_freq:   int   = 200
  train_freq:           int   = 1

  #Reward weights
  reward_vehicle:       float = 5.0
  reward_cost_scale:    float = 1.0
  reward_best_bonus:    float = 2.0
  reward_infeasible:    float = -3.0

  seed:                 int   = 42

CFG = Config()


# Best Known Solutions (BKS) — Solomon benchmark
BKS = {
    'RC101': {'nv': 14, 'td': 1696.94},
    'RC102': {'nv': 12, 'td': 1554.75},
    'RC103': {'nv': 11, 'td': 1261.67},
    'RC104': {'nv': 10, 'td': 1135.48},
    'RC105': {'nv': 13, 'td': 1629.44},
    'RC106': {'nv': 11, 'td': 1424.73},
    'RC107': {'nv': 11, 'td': 1230.48},
    'RC108': {'nv': 10, 'td': 1139.82},
    'RC201': {'nv': 4 , 'td': 1406.94},
    'RC202': {'nv': 3 , 'td': 1365.64},
    'RC203': {'nv': 3 , 'td': 1049.62},
    'RC204': {'nv': 3 , 'td': 798.46} ,
    'RC205': {'nv': 4 , 'td': 1297.65},
    'RC206': {'nv': 3 , 'td': 1146.32},
    'RC207': {'nv': 3 , 'td': 1061.14},
    'RC208': {'nv': 3 , 'td': 828.14} ,
}

print('Config loaded')
print(f'Iterations: {CFG.iterations:,} | Segment: {CFG.segment_size} | Early stop: {CFG.early_stop_patience:,}')

Config loaded
Iterations: 25,000 | Segment: 100 | Early stop: 5,000


# ***3. Data***

In [18]:

class VRPTWInstance:
    def __init__(self, d: Dict):
        self.name          = d['name']
        self.n             = d['n']
        self.capacity      = d['capacity']
        data               = d['data']
        self.coords        = data[:, 1:3]
        self.demands       = data[:, 3]
        self.ready_times   = data[:, 4]
        self.due_times     = data[:, 5]
        self.service_times = data[:, 6]
        diff               = self.coords[:, None, :] - self.coords[None, :, :]
        self.dist          = np.sqrt((diff ** 2).sum(axis=2))
        self.max_dist      = self.dist.max()

def load_rc_datasets(base: str) -> Dict[str, list[VRPTWInstance]]:
    datasets = {}
    for grp in ('rc1', 'rc2'):
        files = sorted(glob.glob(os.path.join(base, f'{grp}*.txt')))
        instances = []
        for path in files:
            with open(path, 'r') as f:
                lines = f.readlines()
            name     = lines[0].strip()
            capacity = float(lines[4].strip().split()[1])
            rows     = [list(map(float, l.split())) for l in lines[9:] if l.strip()]
            data     = np.array(rows)
            d        = {'name': name, 'capacity': capacity, 'data': data, 'n': len(data) - 1}
            instances.append(VRPTWInstance(d))
        datasets[grp] = instances
        print(f'{grp.upper()}: {len(instances)} instances → {[os.path.basename(f) for f in files]}')
    return datasets


DATA_PATH = '/content/solomon/data/Solomon'
DATASETS  = load_rc_datasets(DATA_PATH)
inst      = DATASETS['rc1'][0]
print(f'\nSample: {inst.name} | n={inst.n} | capacity={inst.capacity}')

DATA_PATH = '/content/solomon/data/Solomon'
DATASETS  = load_rc_datasets(DATA_PATH)
inst      = DATASETS['rc1'][0]
print(f'\nSample: {inst.name} | n={inst.n} | capacity={inst.capacity}')

RC1: 8 instances → ['rc101.txt', 'rc102.txt', 'rc103.txt', 'rc104.txt', 'rc105.txt', 'rc106.txt', 'rc107.txt', 'rc108.txt']
RC2: 8 instances → ['rc201.txt', 'rc202.txt', 'rc203.txt', 'rc204.txt', 'rc205.txt', 'rc206.txt', 'rc207.txt', 'rc208.txt']

Sample: RC101 | n=100 | capacity=200.0
RC1: 8 instances → ['rc101.txt', 'rc102.txt', 'rc103.txt', 'rc104.txt', 'rc105.txt', 'rc106.txt', 'rc107.txt', 'rc108.txt']
RC2: 8 instances → ['rc201.txt', 'rc202.txt', 'rc203.txt', 'rc204.txt', 'rc205.txt', 'rc206.txt', 'rc207.txt', 'rc208.txt']

Sample: RC101 | n=100 | capacity=200.0


# ***4. Solution***

In [19]:
#Numba-accelerated route elevation
@njit(cache = True)
def _route_cost(route: np.ndarray, dist: np.ndarray) -> float:
  cost = 0.0
  for i in range(len(route) - 1):
    cost += dist[route[i], route[i + 1]]
  cost += dist[route[-1], 0]
  return cost

@njit(cache=True)
def _route_feasible(route: np.ndarray,
                     demand: np.ndarray,
                     capacity: float,
                     ready_times: np.ndarray,
                     due_times: np.ndarray,
                     service_times: np.ndarray,
                     dist: np.ndarray) -> bool:
  load = 0.0
  for n in route:
    load += demand[n]
    if load > capacity:
      return False

  t = 0.0 # Current time, starting from depot (node 0)
  prev = 0 # Previous node, starting from depot (node 0)

  for n in route:
    arrival_time_at_n = t + dist[prev, n]

    # Check if arrived too late
    if arrival_time_at_n > due_times[n]:
      return False

    # Service starts at max(arrival_time, ready_time)
    service_start_time_at_n = max(arrival_time_at_n, ready_times[n])

    # Check if service starts too late
    if service_start_time_at_n > due_times[n]:
      return False

    # Update current time to departure time from node n
    t = service_start_time_at_n + service_times[n]
    prev = n

  return True


class Solution:
    __slots__ = ('routes', 'inst', '_cost', '_feasible')


    def __init__(self, routes: List[List[int]], inst: VRPTWInstance):
        self.routes    = [r for r in routes if r]
        self.inst      = inst
        self._cost     = None
        self._feasible = None


    @property
    def cost(self) -> float:
        if self._cost is None:
            self._cost = sum(
                _route_cost(np.array(r, dtype=np.int64), self.inst.dist)
                for r in self.routes
            )
        return self._cost

    @property
    def feasible(self) -> bool:
        if self._feasible is None:
            inst = self.inst
            self._feasible = all(
                _route_feasible(
                    np.array(r, dtype=np.int64),
                    inst.demands, inst.capacity,
                    inst.ready_times, inst.due_times,
                    inst.service_times, inst.dist
                )
                for r in self.routes
            )
        return self._feasible

    @property
    def nv(self) -> int:
        return len(self.routes)

    def dominates(self, other: 'Solution') -> bool:
        """True if self is strictly better: fewer vehicles, or same NV but lower cost."""
        if self.nv < other.nv:
            return True
        return self.nv == other.nv and self.cost < other.cost

    def copy(self) -> 'Solution':
        return Solution([r[:] for r in self.routes], self.inst)

    def __repr__(self):
        return f'Solution(nv={self.nv}, cost={self.cost:.2f}, feasible={self.feasible})'


def eval_route(route: List[int], inst: VRPTWInstance) -> Tuple[float, float, bool]:
    """Returns (cost, load, feasible) for a single route."""
    if not route:
        return 0.0, 0.0, True
    arr = np.array(route, dtype=np.int64)
    cost = _route_cost(arr, inst.dist)
    load = sum(inst.demands[n] for n in route)
    ok   = _route_feasible(arr, inst.demands, inst.capacity,
                           inst.ready_times, inst.due_times,
                           inst.service_times, inst.dist)
    return cost, load, ok


print('Solution class ready.')

Solution class ready.


#  ***5. Operators***


In [20]:
# ── DESTROY ──────────────────────────────────────────────────────────────────

def op_random_removal(sol: Solution, size: int) -> Tuple[Solution, List[int]]:
    nodes   = [n for r in sol.routes for n in r]
    removed = random.sample(nodes, min(size, len(nodes)))
    rset    = set(removed)
    sol.routes = [[n for n in r if n not in rset] for r in sol.routes if r]
    sol.routes = [r for r in sol.routes if r]
    sol._cost = sol._feasible = None
    return sol, removed


def op_worst_removal(sol: Solution, size: int) -> Tuple[Solution, List[int]]:
    inst  = sol.inst
    costs = []
    for route in sol.routes:
        for i, node in enumerate(route):
            prev   = route[i - 1] if i > 0 else 0
            nxt    = route[i + 1] if i < len(route) - 1 else 0
            saving = inst.dist[prev, node] + inst.dist[node, nxt] - inst.dist[prev, nxt]
            costs.append((saving, node))
    costs.sort(reverse=True)
    removed = [n for _, n in costs[:size]]
    rset    = set(removed)
    sol.routes = [[n for n in r if n not in rset] for r in sol.routes]
    sol.routes = [r for r in sol.routes if r]
    sol._cost = sol._feasible = None
    return sol, removed


def op_shaw_removal(sol: Solution, size: int) -> Tuple[Solution, List[int]]:
    inst        = sol.inst
    nodes       = [n for r in sol.routes for n in r]
    if not nodes:
        return sol, []
    seed        = random.choice(nodes)
    removed     = [seed]
    removed_set = {seed}
    max_d       = inst.max_dist + 1e-9
    max_tw      = max(inst.due_times - inst.ready_times) + 1e-9
    max_dem     = inst.demands.max() + 1e-9

    while len(removed) < size:
        candidates = [
            (n, 0.6 * inst.dist[seed, n] / max_d
               + 0.3 * abs(inst.ready_times[seed] - inst.ready_times[n]) / max_tw
               + 0.1 * abs(inst.demands[seed]     - inst.demands[n])     / max_dem)
            for n in nodes if n not in removed_set
        ]
        if not candidates:
            break
        removed.append(min(candidates, key=lambda x: x[1])[0])
        removed_set.add(removed[-1])

    rset = set(removed)
    sol.routes = [[n for n in r if n not in rset] for r in sol.routes]
    sol.routes = [r for r in sol.routes if r]
    sol._cost = sol._feasible = None
    return sol, removed


def op_route_removal(sol: Solution, size: int) -> Tuple[Solution, List[int]]:
    if len(sol.routes) <= 1:
        return op_random_removal(sol, size)
    removed, to_rm = [], set()
    for idx, route in sorted(enumerate(sol.routes), key=lambda x: len(x[1])):
        if len(removed) + len(route) <= size * 1.5:
            removed.extend(route)
            to_rm.add(idx)
        if len(removed) >= size:
            break
    sol.routes = [r for i, r in enumerate(sol.routes) if i not in to_rm] or [[]]
    sol._cost = sol._feasible = None
    return sol, removed


# ── REPAIR ───────────────────────────────────────────────────────────────────

def _best_insertion_cost(node: int, route: List[int], inst: VRPTWInstance):
    best_c, best_p = float('inf'), None
    for pos in range(len(route) + 1):
        prev = route[pos - 1] if pos > 0 else 0
        nxt  = route[pos]     if pos < len(route) else 0
        c    = inst.dist[prev, node] + inst.dist[node, nxt] - inst.dist[prev, nxt]
        if c < best_c and eval_route(route[:pos] + [node] + route[pos:], inst)[2]:
            best_c, best_p = c, pos
    return best_c, best_p


def _insert_node(sol: Solution, node: int, inst: VRPTWInstance):
    best_c, best_r, best_p = float('inf'), None, None
    for r_idx, route in enumerate(sol.routes):
        c, p = _best_insertion_cost(node, route, inst)
        if p is not None and c < best_c:
            best_c, best_r, best_p = c, r_idx, p
    if best_r is not None:
        sol.routes[best_r].insert(best_p, node)
    else:
        sol.routes.append([node])
    sol._cost = sol._feasible = None


def op_greedy_insertion(sol: Solution, removed: List[int]) -> Solution:
    inst    = sol.inst
    ordered = sorted(removed, key=lambda n: inst.due_times[n] - inst.ready_times[n])
    for node in ordered:
        _insert_node(sol, node, inst)
    return Solution(sol.routes, inst)


def _regret_insertion(sol: Solution, removed: List[int], k: int) -> Solution:
    inst      = sol.inst
    remaining = removed[:]
    while remaining:
        best_regret, chosen_node, chosen_insert = -float('inf'), None, None
        for node in remaining:
            options = sorted(
                (c, r, p)
                for r, route in enumerate(sol.routes)
                for c, p in [_best_insertion_cost(node, route, inst)]
                if p is not None
            )
            if len(options) >= k:
                regret = sum(options[i][0] - options[0][0] for i in range(1, k))
            elif len(options) >= 2:
                regret = options[1][0] - options[0][0]
            elif len(options) == 1:
                regret = float('inf')
            else:
                regret = -float('inf')
            if regret > best_regret and options:
                best_regret, chosen_node, chosen_insert = regret, node, options[0]
        if chosen_node is not None:
            _, r_idx, pos = chosen_insert
            sol.routes[r_idx].insert(pos, chosen_node)
            sol._cost = sol._feasible = None
            remaining.remove(chosen_node)
        else:
            for node in remaining:
                sol.routes.append([node])
            break
    return Solution(sol.routes, inst)


def op_regret2_insertion(sol: Solution, removed: List[int]) -> Solution:
    return _regret_insertion(sol, removed, k=2)

def op_regret3_insertion(sol: Solution, removed: List[int]) -> Solution:
    return _regret_insertion(sol, removed, k=3)


DESTROY_OPS = [op_random_removal, op_worst_removal, op_shaw_removal, op_route_removal]
REPAIR_OPS  = [op_greedy_insertion, op_regret2_insertion, op_regret3_insertion]
N_D, N_R    = len(DESTROY_OPS), len(REPAIR_OPS)
print(f'Operators: {N_D} destroy × {N_R} repair = {N_D * N_R} combos')


Operators: 4 destroy × 3 repair = 12 combos


## ***6. Initial Solution & Acceptance***


In [21]:
def create_initial(inst: VRPTWInstance) -> Solution:
    """Nearest-feasible greedy, sorted by due time (tightest first)."""
    customers = sorted(range(1, inst.n + 1),
                       key=lambda n: (inst.due_times[n], inst.ready_times[n]))
    unvisited = set(customers)
    routes    = []
    while unvisited:
        route, node, load, t = [], 0, 0.0, 0.0
        while unvisited:
            feasible = [
                n for n in unvisited
                if load + inst.demands[n] <= inst.capacity
                and t + inst.dist[node, n] <= inst.due_times[n]
            ]
            if not feasible:
                break
            nxt = min(feasible, key=lambda n: inst.dist[node, n])
            route.append(nxt)
            unvisited.remove(nxt)
            load += inst.demands[nxt]
            t     = max(t + inst.dist[node, nxt], inst.ready_times[nxt]) + inst.service_times[nxt]
            node  = nxt
        if route:
            routes.append(route)
    sol = Solution(routes, inst)
    assert sol.feasible, 'Initial solution infeasible — check data'
    return sol


def accept(current: Solution, candidate: Solution, temp: float) -> bool:
    if not candidate.feasible:
        return False
    if candidate.nv < current.nv:
        return True
    if candidate.nv == current.nv:
        if candidate.cost < current.cost:
            return True
        delta = candidate.cost - current.cost
        return random.random() < math.exp(-delta / max(temp, 1e-6))
    return False


def get_destroy_size(it: int, cfg: Config, n: int) -> int:
    progress = it / cfg.iterations
    ratio    = cfg.destroy_ratio_max - (cfg.destroy_ratio_max - cfg.destroy_ratio_min) * progress
    return max(3, int(ratio * n))


print('Initial solution builder ready.')

Initial solution builder ready.


# ***7. Standard ALNS (Baseline)***

In [ ]:
class StandardALNS:
    """Adaptive Large Neighbourhood Search with roulette-wheel operator selection."""

    def __init__(self, inst: VRPTWInstance, cfg: Config = CFG):
        self.inst = inst
        self.cfg  = cfg

    def _roulette(self, weights: np.ndarray) -> int:
        probs = weights / weights.sum()
        return int(np.random.choice(len(weights), p=probs))

    def solve(self, seed: int = None, verbose: bool = False) -> Tuple[Solution, List[float]]:
        if seed is not None:
            random.seed(seed); np.random.seed(seed)

        cfg        = self.cfg
        current    = create_initial(self.inst)
        best       = current.copy()
        temp       = cfg.temp_control * current.cost / math.log(2)

        d_weights  = np.ones(N_D)
        r_weights  = np.ones(N_R)
        seg_score  = np.zeros((N_D, N_R))
        seg_count  = np.zeros((N_D, N_R))

        history    = [best.cost]
        no_improve = 0

        for it in range(cfg.iterations):
            d_idx = self._roulette(d_weights)
            r_idx = self._roulette(r_weights)
            size  = get_destroy_size(it, cfg, self.inst.n)

            destroyed, removed = DESTROY_OPS[d_idx](current.copy(), size)
            candidate          = REPAIR_OPS[r_idx](destroyed, removed)

            score = 0
            if accept(current, candidate, temp):
                if candidate.dominates(best):
                    best       = candidate.copy()
                    score      = cfg.sigma1
                    no_improve = 0
                elif candidate.dominates(current):
                    score      = cfg.sigma2
                    no_improve = 0
                else:
                    score      = cfg.sigma3
                    no_improve += 1
                current = candidate
            else:
                no_improve += 1

            seg_score[d_idx, r_idx] += score
            seg_count[d_idx, r_idx] += 1

            # Weight update at segment boundary
            if (it + 1) % cfg.segment_size == 0:
                for d in range(N_D):
                    for r in range(N_R):
                        if seg_count[d, r] > 0:
                            avg          = seg_score[d, r] / seg_count[d, r]
                            d_weights[d] = d_weights[d] * (1 - cfg.weight_decay) + avg * cfg.weight_decay
                            r_weights[r] = r_weights[r] * (1 - cfg.weight_decay) + avg * cfg.weight_decay
                seg_score[:] = 0; seg_count[:] = 0
                d_weights = np.maximum(d_weights, 0.1)
                r_weights = np.maximum(r_weights, 0.1)

            temp *= cfg.temp_decay
            history.append(best.cost)

            if no_improve >= cfg.early_stop_patience:
                if verbose: print(f'  Early stop at iter {it}')
                break

        return best, history


print('StandardALNS ready.')

# Quick sanity check
_inst  = DATASETS['rc1'][0]
_sol, _hist = StandardALNS(_inst, CFG).solve(seed=42)
bks = BKS.get(_inst.name, {})
gap = (_sol.cost - bks.get('td', _sol.cost)) / bks.get('td', _sol.cost) * 100
print(f'{_inst.name}: nv={_sol.nv} (BKS {bks.get("nv","?")}) | cost={_sol.cost:.2f} (BKS {bks.get("td","?")}) | gap={gap:.1f}%')

StandardALNS ready.


# ***8. DQN (Baseline)***

In [ ]:
class DQNNet(nn.Module):
    """Standard DQN: state → Q(s,a) for each operator combo."""
    def __init__(self, state_dim: int, action_dim: int, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )
    def forward(self, x): return self.net(x)


class DoubleDQNNet(nn.Module):
    """Dueling network for Double DQN: V(s) + A(s,a) - mean(A)."""
    def __init__(self, state_dim: int, action_dim: int, hidden: int):
        super().__init__()
        self.feature  = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.LayerNorm(hidden), nn.ReLU()
        )
        self.value    = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 1))
        self.advantage= nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, action_dim))

    def forward(self, x):
        feat = self.feature(x)
        v    = self.value(feat)
        a    = self.advantage(feat)
        return v + a - a.mean(dim=-1, keepdim=True)


class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buf = deque(maxlen=capacity)

    def push(self, s, a, r, ns, done):
        self.buf.append((s, a, r, ns, done))

    def sample(self, k: int):
        batch  = random.sample(self.buf, k)
        s, a, r, ns, d = zip(*batch)
        return (np.array(s, dtype=np.float32),
                np.array(a, dtype=np.int64),
                np.array(r, dtype=np.float32),
                np.array(ns, dtype=np.float32),
                np.array(d, dtype=np.float32))

    def __len__(self): return len(self.buf)


print('RL networks ready. Action space:', N_D * N_R)

In [ ]:
class DQNSolver:
    """
    DQN standalone — builds routes by learning which customer to visit next.
    State: 9 features describing current position + remaining workload.
    Action: next customer to visit (or 0 = return to depot).
    """

    def __init__(self, inst: VRPTWInstance, cfg: Config = CFG):
        self.inst       = inst
        self.cfg        = cfg
        self.state_dim  = 9
        self.action_dim = inst.n + 1   # 0 = depot, 1..n = customers

        self.q   = DQNNet(self.state_dim, self.action_dim, cfg.hidden_dim).to(DEVICE)
        self.q_t = DQNNet(self.state_dim, self.action_dim, cfg.hidden_dim).to(DEVICE)
        self.q_t.load_state_dict(self.q.state_dict())
        self.opt = optim.Adam(self.q.parameters(), lr=cfg.lr)
        self.buf = ReplayBuffer(cfg.buffer_size)
        self.eps = cfg.epsilon_start

    def _state(self, node: int, visited: set, load: float, t: float) -> np.ndarray:
        inst      = self.inst
        unvisited = [n for n in range(1, inst.n + 1) if n not in visited]
        feasible  = sum(
            1 for n in unvisited
            if load + inst.demands[n] <= inst.capacity
            and t + inst.dist[node, n] <= inst.due_times[n]
        )
        return np.array([
            load / inst.capacity,
            t / max(inst.due_times.max(), 1), # Changed inst.horizon to inst.due_times.max()
            len(visited) / inst.n,
            (inst.capacity - load) / inst.capacity,
            len(unvisited) / inst.n,
            feasible / max(len(unvisited), 1),
            inst.coords[node, 0] / 100.0,
            inst.coords[node, 1] / 100.0,
            inst.demands[node]   / inst.capacity,
        ], dtype=np.float32)

    def _feasible_actions(self, node: int, visited: set, load: float, t: float) -> List[int]:
        inst    = self.inst
        actions = [0]   # always can return to depot
        for n in range(1, inst.n + 1):
            if n not in visited:
                if load + inst.demands[n] <= inst.capacity:
                    if t + inst.dist[node, n] <= inst.due_times[n]:
                        actions.append(n)
        return actions

    def _select(self, state: np.ndarray, feasible: List[int]) -> int:
        if random.random() < self.eps:
            return random.choice(feasible)
        with torch.no_grad():
            q = self.q(torch.tensor(state).unsqueeze(0).to(DEVICE)).cpu().numpy()[0]
        return max(feasible, key=lambda a: q[a])

    def _train(self):
        if len(self.buf) < self.cfg.batch_size:
            return
        s, a, r, ns, d = self.buf.sample(self.cfg.batch_size)
        s  = torch.tensor(s).to(DEVICE)
        a  = torch.tensor(a, dtype=torch.long).to(DEVICE)
        r  = torch.tensor(r).to(DEVICE)
        ns = torch.tensor(ns).to(DEVICE)
        d  = torch.tensor(d).to(DEVICE)

        q_pred = self.q(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            q_next = self.q_t(ns).max(1)[0]
            target = r + self.cfg.gamma * q_next * (1 - d)

        loss = F.mse_loss(q_pred, target)
        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 1.0)
        self.opt.step()

    def solve(self, seed: int = None, verbose: bool = False) -> Tuple[Solution, List[float]]:
        if seed is not None:
            random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

        inst          = self.inst
        best_sol      = None
        best_cost     = float('inf')
        history       = []
        self.eps      = self.cfg.epsilon_start
        n_episodes    = self.cfg.iterations // inst.n  # episodes ≈ iterations budget

        for ep in range(n_episodes):
            routes, visited, transitions = [], set(), []

            # Build full solution episode
            while len(visited) < inst.n:
                route, node, load, t = [], 0, 0.0, 0.0

                while True:
                    state   = self._state(node, visited, load, t)
                    feasible = self._feasible_actions(node, visited, load, t)

                    if len(feasible) == 1:  # only depot left
                        break

                    action = self._select(state, feasible)
                    if action == 0:
                        break

                    route.append(action)
                    visited.add(action)

                    dist    = inst.dist[node, action]
                    reward  = -dist / max(inst.max_dist, 1)  # normalized negative distance
                    load   += inst.demands[action]
                    t       = max(t + dist, inst.ready_times[action]) + inst.service_times[action]

                    next_state = self._state(action, visited, load, t)
                    done       = len(visited) == inst.n
                    transitions.append((state, action, reward, next_state, float(done)))
                    node = action

                if route:
                    routes.append(route)

            # End-of-episode bonus reward for solution quality
            sol = Solution(routes, inst)
            if sol.feasible:
                bonus = (best_cost - sol.cost) / max(best_cost, 1) if best_cost < float('inf') else 0
                if transitions:
                    s, a, r, ns, d = transitions[-1]
                    transitions[-1] = (s, a, r + bonus, ns, d)
                if sol.cost < best_cost:
                    best_cost = sol.cost
                    best_sol  = sol.copy()

            self.buf.buf.extend(transitions)

            # Train every 5 episodes
            if ep % 5 == 0:
                for _ in range(min(10, len(self.buf) // self.cfg.batch_size)):
                    self._train()

            # Update target network
            if ep % 20 == 0:
                self.q_t.load_state_dict(self.q.state_dict())

            self.eps = max(self.cfg.epsilon_end, self.eps * self.cfg.epsilon_decay)
            history.append(best_cost if best_cost < float('inf') else 0)

        if best_sol is None:
            best_sol = create_initial(inst)  # fallback nếu không tìm được feasible solution

        return best_sol, history

print('Solving VRPTW with DQNSolver...')
_inst  = DATASETS['rc1'][0]
dqn_solver = DQNSolver(_inst, CFG)
dqn_sol, dqn_hist = dqn_solver.solve(seed=42)

bks = BKS.get(_inst.name, {})
gap = (dqn_sol.cost - bks.get('td', dqn_sol.cost)) / bks.get('td', dqn_sol.cost) * 100
print(f'{_inst.name} (DQN): nv={dqn_sol.nv} (BKS {bks.get("nv","?")}) | cost={dqn_sol.cost:.2f} (BKS {bks.get("td","?")}) | gap={gap:.1f}%')


### ***9. RL-ALNS Solver (DQN & Double DQN)***

In [ ]:
class RLALNSSolver:
    """
    ALNS where operator selection is guided by a DQN agent.
    The RL agent learns which (destroy, repair) pair works best
    given the current search state.

    mode: 'dqn' (standard) | 'double_dqn' (dueling + double update)
    """

    def __init__(self, inst: VRPTWInstance, mode: str = 'dqn', cfg: Config = CFG):
        self.inst       = inst
        self.cfg        = cfg
        self.mode       = mode
        self.action_dim = N_D * N_R

        NetClass  = DoubleDQNNet if mode == 'double_dqn' else DQNNet
        self.q    = NetClass(cfg.state_dim, self.action_dim, cfg.hidden_dim).to(DEVICE)
        self.q_t  = NetClass(cfg.state_dim, self.action_dim, cfg.hidden_dim).to(DEVICE)
        self.q_t.load_state_dict(self.q.state_dict())
        self.opt  = optim.Adam(self.q.parameters(), lr=cfg.lr)
        self.buf  = ReplayBuffer(cfg.buffer_size)
        self.eps  = cfg.epsilon_start

    # ── State ──────────────────────────────────────────────────────────────
    def _state(self, current: Solution, best: Solution,
               it: int, temp: float,
               d_w: np.ndarray, r_w: np.ndarray,
               improvements: deque) -> np.ndarray:
        inst       = self.inst
        cfg        = self.cfg
        imp_rate   = sum(improvements) / len(improvements) if improvements else 0.0
        cost_gap   = min((current.cost - best.cost) / max(best.cost, 1), 1.0)
        nv_ratio   = current.nv / max(self.init_nv, 1)
        progress   = it / cfg.iterations
        stagnation = 1.0 - imp_rate

        lens = [len(r) for r in current.routes] or [0]
        load_vals = [sum(inst.demands[n] for n in r) for r in current.routes] or [0]
        route_bal = float(np.std(lens)  / max(np.mean(lens), 1)) if len(lens) > 1 else 0.0
        load_bal  = float(np.std(load_vals) / max(inst.capacity, 1))

        T0        = cfg.temp_control * best.cost / math.log(2)
        temp_norm = min(temp / max(T0, 1e-6), 1.0)

        dp = d_w / d_w.sum(); rp = r_w / r_w.sum()

        return np.array([
            cost_gap, nv_ratio, progress, imp_rate, stagnation,
            min(route_bal, 1.0), min(load_bal, 1.0), temp_norm,
            dp.max(), dp.min(), rp.max(), rp.min()
        ], dtype=np.float32)

    # ── Action selection ───────────────────────────────────────────────────
    def _act(self, state: np.ndarray) -> int:
        if random.random() < self.eps:
            return random.randrange(self.action_dim)
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            return int(self.q(s).argmax().item())

    def _decode(self, action: int) -> Tuple[int, int]:
        return action // N_R, action % N_R

    # ── Reward ────────────────────────────────────────────────────────────
    def _reward(self, current: Solution, candidate: Solution,
                best: Solution) -> float:
        cfg = self.cfg
        if not candidate.feasible:
            return cfg.reward_infeasible
        r  = (current.nv - candidate.nv) * cfg.reward_vehicle
        r += (current.cost - candidate.cost) / max(current.cost, 1) * 100 * cfg.reward_cost_scale
        if candidate.dominates(best):
            r += cfg.reward_best_bonus
        return float(r)

    # ── Training ──────────────────────────────────────────────────────────
    def _train(self):
        if len(self.buf) < self.cfg.batch_size:
            return
        s, a, r, ns, d = self.buf.sample(self.cfg.batch_size)
        s  = torch.tensor(s,  dtype=torch.float32).to(DEVICE)
        a  = torch.tensor(a,  dtype=torch.long).to(DEVICE)
        r  = torch.tensor(r,  dtype=torch.float32).to(DEVICE)
        ns = torch.tensor(ns, dtype=torch.float32).to(DEVICE)
        d  = torch.tensor(d,  dtype=torch.float32).to(DEVICE)

        q_pred = self.q(s).gather(1, a.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            if self.mode == 'double_dqn':
                # Double DQN: select action with online net, evaluate with target net
                a_next = self.q(ns).argmax(1)
                q_next = self.q_t(ns).gather(1, a_next.unsqueeze(1)).squeeze(1)
            else:
                q_next = self.q_t(ns).max(1)[0]
            target = r + self.cfg.gamma * q_next * (1 - d)

        loss = F.mse_loss(q_pred, target)
        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 1.0)
        self.opt.step()

    # ── Main solve loop ───────────────────────────────────────────────────
    def solve(self, seed: int = None, verbose: bool = False) -> Tuple[Solution, List[float]]:
        if seed is not None:
            random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

        cfg        = self.cfg
        current    = create_initial(self.inst)
        best       = current.copy()
        self.init_nv = current.nv
        temp       = cfg.temp_control * current.cost / math.log(2)

        d_weights  = np.ones(N_D)
        r_weights  = np.ones(N_R)
        seg_score  = np.zeros((N_D, N_R))
        seg_count  = np.zeros((N_D, N_R))
        improvements = deque(maxlen=50)

        history    = [best.cost]
        no_improve = 0
        self.eps   = cfg.epsilon_start

        for it in range(cfg.iterations):
            state  = self._state(current, best, it, temp, d_weights, r_weights, improvements)
            action = self._act(state)
            d_idx, r_idx = self._decode(action)
            size   = get_destroy_size(it, cfg, self.inst.n)

            destroyed, removed = DESTROY_OPS[d_idx](current.copy(), size)
            candidate          = REPAIR_OPS[r_idx](destroyed, removed)

            reward     = self._reward(current, candidate, best)
            alns_score = 0

            if accept(current, candidate, temp):
                improved = candidate.dominates(current)
                improvements.append(1 if improved else 0)
                if candidate.dominates(best):
                    best       = candidate.copy()
                    alns_score = cfg.sigma1
                    no_improve = 0
                elif improved:
                    alns_score = cfg.sigma2
                    no_improve = 0
                else:
                    alns_score = cfg.sigma3
                    no_improve += 1
                current = candidate
            else:
                improvements.append(0)
                no_improve += 1

            # ALNS weight update
            seg_score[d_idx, r_idx] += alns_score
            seg_count[d_idx, r_idx] += 1
            if (it + 1) % cfg.segment_size == 0:
                for d in range(N_D):
                    for r in range(N_R):
                        if seg_count[d, r] > 0:
                            avg          = seg_score[d, r] / seg_count[d, r]
                            d_weights[d] = d_weights[d] * (1 - cfg.weight_decay) + avg * cfg.weight_decay
                            r_weights[r] = r_weights[r] * (1 - cfg.weight_decay) + avg * cfg.weight_decay
                seg_score[:] = 0; seg_count[:] = 0
                d_weights = np.maximum(d_weights, 0.1)
                r_weights = np.maximum(r_weights, 0.1)

            # RL update
            next_state = self._state(current, best, it + 1, temp, d_weights, r_weights, improvements)
            self.buf.push(state, action, reward, next_state, float(it == cfg.iterations - 1))
            if it % cfg.train_freq == 0:
                self._train()
            if it % cfg.target_update_freq == 0:
                self.q_t.load_state_dict(self.q.state_dict())

            self.eps  = max(cfg.epsilon_end, self.eps * cfg.epsilon_decay)
            temp     *= cfg.temp_decay
            history.append(best.cost)

            if no_improve >= cfg.early_stop_patience:
                if verbose: print(f'  Early stop at iter {it}')
                break

        return best, history

    def save(self, path: str):
        save_file({k: v.cpu() for k, v in self.q.state_dict().items()}, path)

    def load(self, path: str):
        self.q.load_state_dict(load_file(path))
        self.q_t.load_state_dict(self.q.state_dict())


print('RLALNSSolver ready.')

### ***10. Experiments***

In [ ]:
ALGORITHMS = {
    'ALNS':       lambda inst, cfg: StandardALNS(inst, cfg),
    'DQN-ALNS':   lambda inst, cfg: RLALNSSolver(inst, mode='dqn',        cfg=cfg),
    'D2QN-ALNS':  lambda inst, cfg: RLALNSSolver(inst, mode='double_dqn', cfg=cfg),
}


def run_instance(inst: VRPTWInstance, cfg: Config, model_dir: str = None,
                 verbose: bool = True) -> Dict:
    bks     = BKS.get(inst.name, {})
    results = {}

    for algo_name, builder in ALGORITHMS.items():
        run_nvs, run_costs, run_times = [], [], []

        for run in range(cfg.n_runs):
            solver = builder(inst, cfg)
            t0     = time.time()
            sol, _ = solver.solve(seed=cfg.seed + run)
            rt     = time.time() - t0

            run_nvs.append(sol.nv)
            run_costs.append(sol.cost)
            run_times.append(rt)

            # Save model
            if model_dir and hasattr(solver, 'save') and run == 0:
                os.makedirs(model_dir, exist_ok=True)
                solver.save(os.path.join(model_dir, f'{algo_name}_{inst.name}.safetensors'))

        avg_nv   = np.mean(run_nvs)
        avg_cost = np.mean(run_costs)
        avg_time = np.mean(run_times)
        gap      = (avg_cost - bks['td']) / bks['td'] * 100 if bks.get('td') else None
        nv_diff  = avg_nv - bks['nv'] if bks.get('nv') else None

        results[algo_name] = {
            'nv': avg_nv, 'cost': avg_cost, 'time': avg_time,
            'gap': gap, 'nv_diff': nv_diff,
            'nv_std': np.std(run_nvs), 'cost_std': np.std(run_costs)
        }
        if verbose:
            print(f'  {algo_name:<12} nv={avg_nv:.1f}  cost={avg_cost:.1f}  '
                  f'gap={gap:.1f}%  time={avg_time:.1f}s')

    return results


def run_all(datasets: Dict, cfg: Config, model_dir: str = '/content/models') -> pd.DataFrame:
    rows = []
    for grp, instances in datasets.items():
        print(f'\n{'='*60}')
        print(f'  Dataset: {grp.upper()} ({len(instances)} instances)')
        print(f'{'='*60}')
        for inst in instances:
            print(f'\n[{inst.name}]')
            res = run_instance(inst, cfg, model_dir=model_dir)
            for algo, metrics in res.items():
                rows.append({
                    'Dataset': grp.upper(), 'Instance': inst.name, 'Algorithm': algo,
                    'NV': round(metrics['nv'], 2), 'TD': round(metrics['cost'], 2),
                    'Gap%': round(metrics['gap'], 2) if metrics['gap'] is not None else None,
                    'NV_diff': round(metrics['nv_diff'], 2) if metrics['nv_diff'] is not None else None,
                    'Time_s': round(metrics['time'], 1),
                    'NV_std': round(metrics['nv_std'], 2), 'TD_std': round(metrics['cost_std'], 2),
                })
    return pd.DataFrame(rows)


print('Experiment runner ready.')

In [ ]:
# ── FULL EXPERIMENT ─────────────────────────────────────────────────────────
# Runtime estimate on Colab T4:
#   25k iters × 3 algos × 3 runs × 16 instances ≈ 8-14 hours
#   Reduce n_runs=1 for faster results, increase for paper

print(f'Starting full experiment | {CFG.iterations:,} iters | {CFG.n_runs} runs/instance')
df = run_all(DATASETS, CFG, model_dir='/content/models')
df.to_csv('/content/results.csv', index=False)
print('\nSaved: /content/results.csv')

In [ ]:
def make_paper_table(df: pd.DataFrame) -> pd.DataFrame:
    """Build summary table grouped by Dataset × Algorithm."""
    summary = (
        df.groupby(['Dataset', 'Algorithm'])
          .agg(NV=('NV', 'mean'), TD=('TD', 'mean'),
               Gap=('Gap%', 'mean'), NV_diff=('NV_diff', 'mean'),
               Time=('Time_s', 'mean'))
          .round(2)
          .reset_index()
    )
    return summary


def print_paper_table(df: pd.DataFrame):
    table = make_paper_table(df)
    print(f'{'Dataset':<8} {'Algorithm':<14} {'NV':>6} {'NV_diff':>8} {'TD':>10} {'Gap%':>7} {'Time':>8}')
    print('-' * 65)
    for _, row in table.iterrows():
        diff = f"{row['NV_diff']:+.2f}" if pd.notna(row['NV_diff']) else '—'
        gap  = f"{row['Gap']:+.2f}%" if pd.notna(row['Gap']) else '—'
        print(f"{row['Dataset']:<8} {row['Algorithm']:<14} {row['NV']:>6.2f} "
              f"{diff:>8} {row['TD']:>10.2f} {gap:>7} {row['Time']:>7.1f}s")


print_paper_table(df)

In [ ]:
COLORS = {'ALNS': '#5f5fae', 'DQN-ALNS': '#d85a30', 'D2QN-ALNS': '#1d9e75'}


def plot_bar_comparison(df: pd.DataFrame, metric: str = 'TD', title: str = None):
    """Bar chart: algorithm comparison per instance, grouped by dataset."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, ds in zip(axes, ['RC1', 'RC2']):
        sub   = df[df['Dataset'] == ds]
        insts = sub['Instance'].unique()
        algos = list(COLORS.keys())
        x     = np.arange(len(insts))
        w     = 0.25
        for j, algo in enumerate(algos):
            vals = [sub[sub['Instance'] == i][metric].mean() for i in insts]
            ax.bar(x + j * w, vals, w, label=algo, color=COLORS[algo], alpha=0.85, edgecolor='white')
        ax.set_xticks(x + w)
        ax.set_xticklabels([i[-3:] for i in insts], fontsize=9)
        ax.set_title(f'{ds} — {metric}', fontsize=12, fontweight='bold')
        ax.set_ylabel(metric)
        ax.grid(axis='y', alpha=0.3)
        ax.legend()
    plt.suptitle(title or f'{metric} by Instance & Algorithm', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'/content/{metric}_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_convergence(inst: VRPTWInstance, cfg: Config, seed: int = 42):
    """Convergence curves for all algorithms on one instance."""
    fig, ax = plt.subplots(figsize=(10, 5))
    for algo_name, builder in ALGORITHMS.items():
        solver = builder(inst, cfg)
        _, hist = solver.solve(seed=seed)
        ax.plot(hist, label=algo_name, color=COLORS[algo_name], alpha=0.85, linewidth=1.5)
    if inst.name in BKS:
        ax.axhline(BKS[inst.name]['td'], color='black', linestyle='--',
                   linewidth=1, label=f'BKS ({BKS[inst.name]["td"]:.1f})')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Best Cost')
    ax.set_title(f'Convergence — {inst.name}', fontsize=12, fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'/content/convergence_{inst.name}.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_solution(sol: Solution, title: str = 'Solution'):
    """Visualize route plan on 2D map."""
    inst   = sol.inst
    depot  = inst.coords[0]
    cmap   = plt.cm.Set3(np.linspace(0, 1, max(sol.nv, 1)))
    fig, ax = plt.subplots(figsize=(10, 8))
    for idx, route in enumerate(sol.routes):
        pts = np.array([depot] + [inst.coords[n] for n in route] + [depot])
        ax.plot(pts[:, 0], pts[:, 1], '-o', color=cmap[idx],
                linewidth=1.5, markersize=5, alpha=0.8)
    ax.scatter(*depot, c='red', s=200, marker='*', zorder=5, label='Depot')
    ax.set_title(f'{title}\nnv={sol.nv}  cost={sol.cost:.2f}  feasible={sol.feasible}',
                 fontsize=11)
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    plt.show()


# Plot after experiments
plot_bar_comparison(df, metric='TD', title='Total Distance — All Instances')
plot_bar_comparison(df, metric='NV', title='Number of Vehicles — All Instances')
plot_convergence(DATASETS['rc1'][0], CFG)

In [ ]:
def load_and_run(model_path: str, inst: VRPTWInstance, mode: str = 'double_dqn',
                 cfg: Config = CFG, max_vehicles: int = None) -> Solution:
    """Load trained model, run inference, optionally cap vehicle count."""
    solver = RLALNSSolver(inst, mode=mode, cfg=cfg)
    solver.load(model_path)
    solver.eps = 0.0      # pure exploitation
    sol, _    = solver.solve(seed=0)

    if max_vehicles and sol.nv > max_vehicles:
        sol = _merge_routes(sol, inst, max_vehicles)

    return sol


def _merge_routes(sol: Solution, inst: VRPTWInstance, target_nv: int) -> Solution:
    """Greedily merge smallest routes until nv <= target_nv."""
    routes = [r[:] for r in sol.routes]
    while len(routes) > target_nv:
        routes_load = [(i, r, sum(inst.demands[n] for n in r)) for i, r in enumerate(routes)]
        routes_load.sort(key=lambda x: x[2])
        i1, r1, _ = routes_load[0]
        i2, r2, _ = routes_load[1]
        merged = r1 + r2
        if sum(inst.demands[n] for n in merged) > inst.capacity:
            break
        test = Solution([[n for i, r in enumerate(routes)
                          for n in r if i not in {i1, i2}]] + [merged], inst)
        # Rebuild properly
        new_routes = [r for i, r in enumerate(routes) if i not in {i1, i2}] + [merged]
        test = Solution(new_routes, inst)
        if not test.feasible:
            break
        routes = [r for i, r in enumerate(routes) if i not in {i1, i2}] + [merged]
    return Solution(routes, inst)


# Example inference
# sol = load_and_run('/content/models/D2QN-ALNS_RC101.safetensors',
#                    DATASETS['rc1'][0], mode='double_dqn', max_vehicles=10)
# plot_solution(sol, 'D2QN-ALNS — RC101 (max 10 vehicles)')

print('Inference functions ready.')

In [ ]:
# Per-instance detailed CSV
df.to_csv('/content/results_full.csv', index=False)

# Summary table CSV (for LaTeX)
make_paper_table(df).to_csv('/content/results_summary.csv', index=False)

# Print LaTeX-ready table (manual cleanup needed)
summary = make_paper_table(df)
print(summary.to_latex(index=False, float_format='%.2f',
                       caption='Comparison of ALNS, DQN-ALNS, and D2QN-ALNS on RC benchmarks',
                       label='tab:results'))